# 법령 검색 성능 평가

이미 만든 법령 벡터 165개(법령 95건)를 그대로 쓰고, 평가 질의 24개만 새로 인코딩해 **Hit@1 / Hit@3 / MRR@3**을 측정합니다.

- 모델 재학습·파인튜닝은 하지 않습니다. 질의 24개만 인코딩하므로 CPU로도 돌아갑니다.
- 정답 조문은 작성자가 지정한 **기대 근거**이며 법률 전문가 검수를 거치지 않았습니다.
- 점수는 코사인 유사도이고 위험확률이 아닙니다.

`런타임 > 모두 실행`으로 돌리면 됩니다.

## 1. 벡터·메타데이터·평가질의 복원
노트북에 포함된 데이터를 풀고 형상·정규화·정답키 존재를 검증합니다.

In [ ]:
import base64, json, hashlib, io
import numpy as np

VEC_B64 = "..."
META_B64 = "..."
EVAL_B64 = "WwogIHsKICAgICJpZCI6ICJRMDEiLAogICAgInF1ZXJ5IjogIuyehOywqOyduOydgCDqs4Tslb0g6riw6rCEIOykkSDrsJzsg53tlZjripQg66qo65OgIOyImOumrOu5hOyZgCDsiJjshKDruYTrpbwg67aA64u07ZWc64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19DSVZJTF82MjMiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfQ0lWSUxfNjI2IgogICAgXSwKICAgICJpc3N1ZSI6ICLsiJjshKDsnZjrrLQg7KCE6rCAIgogIH0sCiAgewogICAgImlkIjogIlEwMiIsCiAgICAicXVlcnkiOiAi7J6E64yA7J247J2AIOyDiOuhnOyatCDsnoTssKjsnbjsnbQg6rWs7ZW07KeEIO2bhOyXkCDrs7Tspp3quIjsnYQg67CY7ZmY7ZWc64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19DSVZJTF81MzYiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfSE9VU0lOR180IgogICAgXSwKICAgICJpc3N1ZSI6ICLrs7Tspp3quIgg67CY7ZmYIOyngOyXsCIKICB9LAogIHsKICAgICJpZCI6ICJRMDMiLAogICAgInF1ZXJ5IjogIuywqOyehOydhCAx7ZqM652864+EIOyXsOyytO2VmOuptCDsnoTrjIDsnbjsnYAg7KaJ7IucIOqzhOyVveydhCDtlbTsp4DtlaAg7IiYIOyeiOuLpC4iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfQ0lWSUxfNjQwIgogICAgXSwKICAgICJhbHNvIjogW10sCiAgICAiaXNzdWUiOiAi7Jew7LK0IO2VtOyngCDquLDspIAiCiAgfSwKICB7CiAgICAiaWQiOiAiUTA0IiwKICAgICJxdWVyeSI6ICLrrLXsi5zsoIHsnLzroZwg6rCx7Iug65CcIOyehOuMgOywqOulvCDsnoTssKjsnbjsnbQg7ZW07KeAIO2GteyngO2VmOuptCDslrjsoJwg7Zqo66Cl7J20IOyDneq4sOuCmOyalD8iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfSE9VU0lOR182XzIiCiAgICBdLAogICAgImFsc28iOiBbXSwKICAgICJpc3N1ZSI6ICLrrLXsi5zsoIEg6rCx7IugIO2VtOyngCIKICB9LAogIHsKICAgICJpZCI6ICJRMDUiLAogICAgInF1ZXJ5IjogIuyehOuMgOyduOydgCDtlYTsmpTtlZjri6Tqs6Ag7YyQ64uo7ZWY66m0IOyWuOygnOuToOyngCDssKjsnoTsnYQg7J247IOB7ZWgIOyImCDsnojri6QuIiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0hPVVNJTkdfNyIKICAgIF0sCiAgICAiYWxzbyI6IFsKICAgICAgIkxBV19IT1VTSU5HX0RFQ1JFRV84IgogICAgXSwKICAgICJpc3N1ZSI6ICLssKjsnoQg7Kad7JWhIOygnO2VnCIKICB9LAogIHsKICAgICJpZCI6ICJRMDYiLAogICAgInF1ZXJ5IjogIuyehOywqOyduOydgCDsoITsnoXsi6Dqs6DsmYAg7ZmV7KCV7J287J6Q66W8IOuwm+yngCDslYrquLDroZwg7ZWc64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19IT1VTSU5HXzMiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfSE9VU0lOR18zXzIiLAogICAgICAiTEFXX0hPVVNJTkdfMTAiCiAgICBdLAogICAgImlzc3VlIjogIuuMgO2VreugpSDtj6zquLAiCiAgfSwKICB7CiAgICAiaWQiOiAiUTA3IiwKICAgICJxdWVyeSI6ICLsnoTssKjsnbjsnYAg7Ye06rGw7ZWgIOuVjCDrj4TrsLDsmYAg7J6l7YyQ7J2EIOyDiOqyg+ycvOuhnCDqtZDssrTtlZjsl6wg7JuQ7IOB67O16rWs7ZWc64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19DSVZJTF82MTUiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfQ0lWSUxfNjU0IgogICAgXSwKICAgICJpc3N1ZSI6ICLsm5Dsg4Htmozrs7Ug67KU7JyEIgogIH0sCiAgewogICAgImlkIjogIlEwOCIsCiAgICAicXVlcnkiOiAi67O07Kad6riI7J2YIOydvOu2gOulvCDsm5TshLjroZwg7KCE7ZmY7ZWgIOuVjCDsoIHsmqnrkJjripQg7IKw7KCV66WgIOyDge2VnOydgCDslrzrp4jsnbjqsIDsmpQ/IiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0hPVVNJTkdfN18yIgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX0hPVVNJTkdfREVDUkVFXzkiCiAgICBdLAogICAgImlzc3VlIjogIuyblOywqOyehCDsoITtmZjsnKgiCiAgfSwKICB7CiAgICAiaWQiOiAiUTA5IiwKICAgICJxdWVyeSI6ICLshozslaHsnoTssKjsnbjsnbQg7Jqw7ISg67OA7KCc67Cb7J2EIOyImCDsnojripQg67O07Kad6riIIOykkSDsnbzsoJXslaHsnZgg67KU7JyE6rCAIOq2geq4iO2VqeuLiOuLpC4iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfSE9VU0lOR19ERUNSRUVfMTAiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfSE9VU0lOR184IiwKICAgICAgIkxBV19IT1VTSU5HX0RFQ1JFRV8xMSIKICAgIF0sCiAgICAiaXNzdWUiOiAi7LWc7Jqw7ISg67OA7KCcIgogIH0sCiAgewogICAgImlkIjogIlExMCIsCiAgICAicXVlcnkiOiAi67O07Kad6riI7J2EIOuPjOugpOuwm+yngCDrqrvtlZwg7LGEIOydtOyCrO2VtOyVvCDtlZjripTrjbAg7J6E7LCo6raM65Ox6riw66qF66C57J2EIOyLoOyyre2VmOqzoCDsi7bsirXri4jri6QuIiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0hPVVNJTkdfM18zIgogICAgXSwKICAgICJhbHNvIjogW10sCiAgICAiaXNzdWUiOiAi7J6E7LCo6raM65Ox6riw66qF66C5IgogIH0sCiAgewogICAgImlkIjogIlExMSIsCiAgICAicXVlcnkiOiAi7J6E7LCo7J247J20IOyepeq4sOyImOyEoOy2qeuLueq4iOydhCDrjIDsi6Ag64Kp67aA7ZaI64qU642wIOuPjOugpOuwm+ydhCDsiJgg7J6I64KY7JqUPyIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19BUFRfREVDUkVFXzMxIgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX0FQVF8zMCIKICAgIF0sCiAgICAiaXNzdWUiOiAi7J6l6riw7IiY7ISg7Lap64u56riIIOuwmO2ZmCIKICB9LAogIHsKICAgICJpZCI6ICJRMTIiLAogICAgInF1ZXJ5IjogIuyehOywqOyduOydgCDsnoTrjIDsnbjsnZgg64+Z7J2YIOyXhuydtCDsoITrjIDtlaAg7IiYIOyXhuycvOupsCDsnITrsJjtlZjrqbQg6rOE7JW97J2EIO2VtOyngO2VnOuLpC4iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfQ0lWSUxfNjI5IgogICAgXSwKICAgICJhbHNvIjogW10sCiAgICAiaXNzdWUiOiAi7KCE64yAIOygnO2VnCIKICB9LAogIHsKICAgICJpZCI6ICJRMTMiLAogICAgInF1ZXJ5IjogIuynkeyjvOyduOydtCDsnpDsi6DsnbQg7KeB7KCRIOqxsOyjvO2VmOqyoOuLpOupsCDqs4Tslb3qsLHsi6Ag7JqU6rWs66W8IOqxsOygiO2WiOyKteuLiOuLpC4iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfSE9VU0lOR182XzMiCiAgICBdLAogICAgImFsc28iOiBbXSwKICAgICJpc3N1ZSI6ICLqsLHsi6DqsbDsoIgg7IKs7JygIgogIH0sCiAgewogICAgImlkIjogIlExNCIsCiAgICAicXVlcnkiOiAi7J20IOqzhOyVveyXkCDqtIDtlZwg7IaM7Iah7J2AIOyehOuMgOyduCDso7zshozsp4Ag67KV7JuQ7J2EIOq0gO2VoOuhnCDtlZzri6QuIiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0NJVklMX1BST0NFRFVSRV8yOSIKICAgIF0sCiAgICAiYWxzbyI6IFsKICAgICAgIkxBV19URVJNU18xNCIKICAgIF0sCiAgICAiaXNzdWUiOiAi6rSA7ZWgIO2VqeydmCIKICB9LAogIHsKICAgICJpZCI6ICJRMTUiLAogICAgInF1ZXJ5IjogIuyehOuMgOyduOydgCDsnoTrjIAg66qp7KCB66y87J2YIO2VmOyekOyXkCDrjIDtlZjsl6wg7Ja065ag7ZWcIOyxheyehOuPhCDsp4Dsp4Ag7JWK64qU64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19URVJNU183IgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX0NJVklMXzYyMyIKICAgIF0sCiAgICAiaXNzdWUiOiAi66m07LGF7KGw7ZWtIgogIH0sCiAgewogICAgImlkIjogIlExNiIsCiAgICAicXVlcnkiOiAi6rOE7JW97J2EIOychOuwmO2VmOuptCDsnoTssKjsnbjsnbQg64K4IOuztOymneq4iCDsoITslaHsnYQg7JyE7JW96riI7Jy866GcIOuqsOy3qO2VnOuLpC4iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfQ0lWSUxfMzk4IgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX1RFUk1TXzgiCiAgICBdLAogICAgImlzc3VlIjogIuychOyVveq4iCDqs7zri6QiCiAgfSwKICB7CiAgICAiaWQiOiAiUTE3IiwKICAgICJxdWVyeSI6ICLsnoTssKjsnbjsl5Dqsowg67aI66as7ZWcIO2KueyVveydgCDtmqjroKXsnbQg7JeG64uk6rOgIO2VmOuKlOuNsCDqt7zqsbAg7KGw66y47J20IOustOyXh+yduOqwgOyalD8iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfSE9VU0lOR18xMCIKICAgIF0sCiAgICAiYWxzbyI6IFsKICAgICAgIkxBV19DSVZJTF82NTIiCiAgICBdLAogICAgImlzc3VlIjogIuqwle2Wieq3nOyglSIKICB9LAogIHsKICAgICJpZCI6ICJRMTgiLAogICAgInF1ZXJ5IjogIuyehOuMgOywqCDquLDqsITsnYQgMeuFhOycvOuhnCDsoJXtlojripTrjbAg7J6E7LCo7J247J20IDLrhYTsnYQg7KO87J6l7ZWgIOyImCDsnojrgpjsmpQ/IiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0hPVVNJTkdfNCIKICAgIF0sCiAgICAiYWxzbyI6IFtdLAogICAgImlzc3VlIjogIuy1nOuLqCDsobTsho3quLDqsIQiCiAgfSwKICB7CiAgICAiaWQiOiAiUTE5IiwKICAgICJxdWVyeSI6ICLqs7XsnbjspJHqsJzsgqzqsIAg7KSR6rCcIOqzvOygleyXkOyEnCDqsbTrrLzsnZgg6raM66as6rSA6rOE66W8IOyEpOuqhe2VtCDso7zsp4Ag7JWK7JWY7Iq164uI64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19CUk9LRVJfMjUiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfQlJPS0VSX0RFQ1JFRV8yMSIKICAgIF0sCiAgICAiaXNzdWUiOiAi7KSR6rCc7IKsIO2ZleyduOyEpOuqhSIKICB9LAogIHsKICAgICJpZCI6ICJRMjAiLAogICAgInF1ZXJ5IjogIuuTseq4sOu2gOyDgSDshozsnKDsnpDqsIAg7Iug7YOB7ZqM7IKs7J24642wIOyehOuMgOywqOqzhOyVveydgCDriITqtazsmYAg7LK06rKw7ZW07JW8IO2VmOuCmOyalD8iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfVFJVU1RfMzEiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfVFJVU1RfNCIsCiAgICAgICJMQVdfUkVHSVNUUllfODEiCiAgICBdLAogICAgImlzc3VlIjogIuyLoO2DgSDrtoDrj5nsgrAiCiAgfSwKICB7CiAgICAiaWQiOiAiUTIxIiwKICAgICJxdWVyeSI6ICLsnoTrjIDsnbjsnbQg67O07Kad6riIIOuwmO2ZmOydhCDsp4DssrTtlbTshJwg7IOd6ri0IOyGkO2VtOulvCDrsLDsg4HrsJvsnYQg7IiYIOyeiOuCmOyalD8iLAogICAgImdvbGQiOiBbCiAgICAgICJMQVdfQ0lWSUxfMzkwIgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX0NJVklMXzM5MyIKICAgIF0sCiAgICAiaXNzdWUiOiAi7LGE66y067aI7J207ZaJIOyGkO2VtOuwsOyDgSIKICB9LAogIHsKICAgICJpZCI6ICJRMjIiLAogICAgInF1ZXJ5IjogIuyehOywqOyduOydmCDqs7zsi6TroZwg67aI7J20IOuCmCDsmIbsp5HquYzsp4Ag67KI7KGM7J2EIOuVjCDrsLDsg4HslaHsnbQg7KSE7Ja065OkIOyImCDsnojrgpjsmpQ/IiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0ZJUkVfMyIKICAgIF0sCiAgICAiYWxzbyI6IFsKICAgICAgIkxBV19GSVJFXzIiCiAgICBdLAogICAgImlzc3VlIjogIuyLpO2ZlCDssYXsnoQiCiAgfSwKICB7CiAgICAiaWQiOiAiUTIzIiwKICAgICJxdWVyeSI6ICLqtIDrpqzruYQg7ZWt66qp67OEIOuCtOyXreydhCDqs7XqsJztlZjsp4Ag7JWK6rOgIOyehOuMgOyduOydtCDsnoTsnZjroZwg7LGF7KCV7ZWc64ukLiIsCiAgICAiZ29sZCI6IFsKICAgICAgIkxBV19BUFRfMjMiCiAgICBdLAogICAgImFsc28iOiBbCiAgICAgICJMQVdfQVBUX0RFQ1JFRV8yMyIKICAgIF0sCiAgICAiaXNzdWUiOiAi6rSA66as67mEIOqzteqwnCIKICB9LAogIHsKICAgICJpZCI6ICJRMjQiLAogICAgInF1ZXJ5IjogIuyehOuMgOywqOqzhOyVveydhCDrp7rquLAg7KCE7JeQIOyehOuMgOyduOydtCDshKDsiJzsnIQg67O07Kad6riI6rO8IOyytOuCqSDshLjquIgg7KCV67O066W8IOyVjOugpOykmOyVvCDtlZjrgpjsmpQ/IiwKICAgICJnb2xkIjogWwogICAgICAiTEFXX0hPVVNJTkdfM183IgogICAgXSwKICAgICJhbHNvIjogWwogICAgICAiTEFXX0hPVVNJTkdfM182IgogICAgXSwKICAgICJpc3N1ZSI6ICLsnoTrjIDsnbgg7KCV67O0IOygnOyLnCIKICB9Cl0="

vec_bytes = base64.b64decode(VEC_B64)
print("벡터 sha256:", hashlib.sha256(vec_bytes).hexdigest())
vectors = np.load(io.BytesIO(vec_bytes), allow_pickle=False)
meta = [json.loads(l) for l in base64.b64decode(META_B64).decode("utf-8").splitlines() if l.strip()]
EVALSET = json.loads(base64.b64decode(EVAL_B64).decode("utf-8"))

assert vectors.shape == (165, 1024), vectors.shape
assert vectors.dtype == np.float32 and np.isfinite(vectors).all()
assert len(meta) == 165 and [m["vector_row"] for m in meta] == list(range(165))
norms = np.linalg.norm(vectors, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5), (float(norms.min()), float(norms.max()))

DOC_KEYS = sorted({m["document_key"] for m in meta})
assert len(DOC_KEYS) == 95, len(DOC_KEYS)
known = set(DOC_KEYS)
for q in EVALSET:
    missing = [k for k in q["gold"] + q["also"] if k not in known]
    assert not missing, (q["id"], missing)
    assert q["gold"], q["id"]

DOC_INFO = {}
for m in meta:
    DOC_INFO.setdefault(m["document_key"], {"law_name": m["law_name"], "article": m["article"],
                                            "article_title": m["article_title"], "source_url": m["source_url"]})

print("문서", len(DOC_KEYS), "청크", len(meta), "질의", len(EVALSET))
print("정답 조문은 작성자가 지정한 기대 근거이며 최종 법률 판단이 아닙니다.")

## 2. 질의 임베딩
문서 벡터와 같은 모델·revision·CLS 풀링·L2 정규화를 사용합니다.

In [ ]:
import time
import importlib.metadata
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "BAAI/bge-m3"
REVISION = "3069def033ce91d907258f9a830e442610dbfe0b"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, use_fast=True)
assert tok.is_fast
model = AutoModel.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=dtype).to(device).eval()
assert int(model.config.hidden_size) == 1024
print("모델 로딩 완료:", device, dtype, flush=True)


def embed(texts):
    out = []
    for i in range(0, len(texts), 8):
        batch = texts[i:i + 8]
        enc = tok(batch, padding=True, truncation=False, return_tensors="pt")
        assert enc["input_ids"].shape[1] <= 8192
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.inference_mode():
            h = model(**enc).last_hidden_state
        out.append(F.normalize(h[:, 0, :].float(), p=2, dim=1).cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0)


t0 = time.time()
queries = [q["query"] for q in EVALSET]
qvecs = embed(queries)
assert qvecs.shape == (len(EVALSET), 1024) and np.isfinite(qvecs).all()
assert np.allclose(np.linalg.norm(qvecs, axis=1), 1.0, atol=1e-5)
print("질의 임베딩 완료", round(time.time() - t0, 2), "초", flush=True)

probe = embed([queries[0]])[0]
print("같은 질의 재인코딩 코사인:", round(float(probe @ qvecs[0]), 6))

## 3. 지표 계산
청크 점수를 문서 단위로 집계(max/mean/sum)해 순위를 만듭니다. strict는 1순위 정답만, lenient는 대체 근거까지 인정합니다.

In [ ]:
def rank_docs(scores, mode):
    agg = {}
    for i, m in enumerate(meta):
        agg.setdefault(m["document_key"], []).append(float(scores[i]))
    if mode == "max":
        table = {d: max(v) for d, v in agg.items()}
    elif mode == "mean":
        table = {d: float(np.mean(v)) for d, v in agg.items()}
    elif mode == "sum":
        table = {d: float(np.sum(v)) for d, v in agg.items()}
    else:
        raise ValueError(mode)
    order = sorted(table.items(), key=lambda kv: (-kv[1], kv[0]))
    return [d for d, _ in order], table


def first_hit_rank(order, targets):
    for i, d in enumerate(order, start=1):
        if d in targets:
            return i
    return None


def evaluate(mode, lenient):
    ranks, rows = [], []
    for qi, q in enumerate(EVALSET):
        scores = vectors @ qvecs[qi]
        assert np.isfinite(scores).all()
        order, table = rank_docs(scores, mode)
        targets = set(q["gold"]) | (set(q["also"]) if lenient else set())
        r = first_hit_rank(order, targets)
        ranks.append(r)
        rows.append({
            "id": q["id"], "query": q["query"], "issue": q["issue"],
            "gold": q["gold"], "also": q["also"], "gold_rank": r,
            "gold_score": round(max(table[g] for g in q["gold"]), 4),
            "top5": [{"doc": d, "law": DOC_INFO[d]["law_name"], "article": DOC_INFO[d]["article"],
                      "title": DOC_INFO[d]["article_title"], "score": round(table[d], 4)} for d in order[:5]],
        })
    n = len(ranks)

    def hit(k):
        return sum(1 for r in ranks if r is not None and r <= k) / n

    def mrr(k):
        return sum((1.0 / r) if (r is not None and r <= k) else 0.0 for r in ranks) / n

    summary = {"aggregation": mode, "gold_mode": "lenient" if lenient else "strict", "queries": n,
               "hit@1": round(hit(1), 4), "hit@3": round(hit(3), 4), "hit@5": round(hit(5), 4),
               "mrr@3": round(mrr(3), 4), "mrr@10": round(mrr(10), 4),
               "miss@3": [rows[i]["id"] for i, r in enumerate(ranks) if r is None or r > 3]}
    return summary, rows


VARIANTS = [(m, l) for m in ("max", "mean", "sum") for l in (False, True)]
all_summaries = [evaluate(m, l)[0] for m, l in VARIANTS]
for s in all_summaries:
    print(s["aggregation"], s["gold_mode"], "Hit@1", s["hit@1"], "Hit@3", s["hit@3"], "MRR@3", s["mrr@3"])

main_summary, main_rows = evaluate("max", False)
print()
print("== 기본 설정(문서별 최고 점수, 엄격 정답) ==")
print(json.dumps(main_summary, ensure_ascii=False, indent=2))

## 4. 결과 저장과 실패 사례

In [ ]:
import csv, zipfile, datetime
from pathlib import Path

outdir = Path("eval_output")
outdir.mkdir(exist_ok=True)
np.save(outdir / "query_vectors.npy", qvecs, allow_pickle=False)

lenient_summary, lenient_rows = evaluate("max", True)
report = {
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "embedding_model": MODEL_ID, "revision": REVISION,
    "device": str(device), "model_dtype": str(dtype),
    "transformers_version": importlib.metadata.version("transformers"),
    "torch_version": torch.__version__,
    "documents": 95, "chunks": 165, "queries": len(EVALSET),
    "ranking_unit": "document (청크 점수를 문서 단위로 집계)",
    "variants": all_summaries, "primary": main_summary, "lenient": lenient_summary,
    "gold_labels_note": "정답 조문은 작성자가 지정한 기대 근거이며 법률 전문가 검수를 거치지 않았습니다.",
    "score_note": "점수는 코사인 유사도이며 위험확률이 아닙니다.",
    "model_finetuned": False,
}
(outdir / "retrieval_eval_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(outdir / "retrieval_eval_details.json").write_text(json.dumps({"strict": main_rows, "lenient": lenient_rows}, ensure_ascii=False, indent=2), encoding="utf-8")

with (outdir / "retrieval_eval_results.csv").open("w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "query", "issue", "gold", "gold_rank", "gold_score", "top1", "top1_score", "top2", "top3", "top4", "top5"])
    for r in main_rows:
        t = r["top5"]
        w.writerow([r["id"], r["query"], r["issue"], ";".join(r["gold"]), r["gold_rank"], r["gold_score"],
                    t[0]["law"] + " " + t[0]["article"], t[0]["score"]] + [x["law"] + " " + x["article"] for x in t[1:5]])

names = ["query_vectors.npy", "retrieval_eval_summary.json", "retrieval_eval_details.json", "retrieval_eval_results.csv"]
zip_path = Path("retrieval_eval_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for n in names:
        z.write(outdir / n, n)
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None

print("저장 완료:", names)
print()
print("== 정답 근거가 상위 3위 밖인 질의 ==")
for r in main_rows:
    if r["gold_rank"] is None or r["gold_rank"] > 3:
        print(r["id"], r["query"])
        print("   기대:", r["gold"], "/ 실제 순위:", r["gold_rank"], "/ 점수:", r["gold_score"])
        print("   1위:", r["top5"][0]["law"], r["top5"][0]["article"], r["top5"][0]["score"])

## 5. 내려받기

In [ ]:
from google.colab import files
files.download("retrieval_eval_output.zip")